# Solar Filament Segmentation Challenge 2026 — Reproducible Offline Multi-Fold Pipeline (Paradigm v10)

This self-contained notebook reproduces all test predictions strictly **offline** (`enable_internet: false`, `pretrained=False`) from stored `fp16` ensemble weights and the calibrated 24-feature meta-scorer JSON, operating solely on raw 8-bit grayscale H-Alpha GONG observations (`2048 x 2048`).

## Architecture & Methodology
1. **Physics-Informed 5-Channel Native 1:1 Representation (`physics_v2`)**
   - Fits sub-pixel solar disk center `(cx, cy)` and radius `R_sun` via IRLS circle regression on the bright chromospheric limb quantile.
   - Computes radial limb-darkening flattening (`I_flat`), bright-plage-clamped multi-scale dark depth (`D_multi` across Gaussian/morphology scales `[7, 15, 29, 55]`), and 8-orientation anisotropic ribbonness (`T_ribbon`) within the native `1888 x 1888` active-disk integer slice (zero resize blur), plus on-GPU local `15 x 15` fine barb contrast and heliocentric cosine `mu_disk = sqrt(max(0, 1 - rho^2))`.
2. **Stage-1 Complementary 1408 px Mask R-CNN v2 Instance Detector**
   - ResNet50-FPN v2 with filament-calibrated aspect ratios `(0.15, 0.33, 0.6, 1.0, 1.67, 3.0, 6.5)` and `28 x 28` RoIAlign mask heads.
   - Multi-fold `4x` flip TTA proposals clustered globally (`iou_thr=0.45`) for instance separation and false-positive screening.
3. **Heterogeneous Cooperative 1:1 Co-DeepNet (ConvNeXt-Small + StripBarbUNet)**
   - **Peer A**: Native stride-1 boundary stem fused with a ConvNeXt-Small hierarchical encoder across scales `1/16 -> 1/1`.
   - **Peer B**: Full-resolution anisotropic `(3x3, 1x9, 9x1, dilated-3x3)` `RibbonBarbBlock` U-Net preserving fine barbs and thin filament channels.
   - **CrossBridge**: Bidirectional `tanh`-gated `CrossBridge` feature exchanges at `1/16, 1/8, 1/4, 1/2` plus fused decoder (`head_a`, `head_b`, `head_f`), averaged across all fold models with `4x` flip TTA (`1888 x 1888`).
4. **Geodesic Hysteresis & 24-Feature Expected-PQ Meta-Scorer**
   - Dual-threshold seed-to-low geodesic propagation (`scipy.ndimage.binary_propagation`), detector-guided touch splitting, waist-only collinear fragment healing, and 5-subfold chronological Ridge + L2-Logistic expected-PQ scoring (`metascorer_v10.json`), followed by strict zero-overlap compositing and compressed COCO RLE encoding.


In [ ]:
"""Cell 1: Environment setup, deterministic imports, and COCO RLE utilities (with offline fallback)."""
from __future__ import annotations

import csv
import copy
import gc
import json
import math
import os
import re
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import cv2
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.rpn import RPNHead
from torchvision.ops import boxes as box_ops

try:
    from pycocotools import mask as mu
    HAS_PYCOCOTOOLS = True
except Exception:
    HAS_PYCOCOTOOLS = False

    def _encode_rle_pure(mask_u8: np.ndarray) -> str:
        flat = np.asarray(mask_u8, dtype=np.uint8).reshape(-1, order="F")
        n = flat.size
        if n == 0:
            return ""
        diffs = np.flatnonzero(flat[1:] != flat[:-1]) + 1
        starts = np.concatenate(([0], diffs))
        ends = np.concatenate((diffs, [n]))
        lengths = (ends - starts).tolist()
        if flat[0] == 1:
            lengths = [0] + lengths
        out = []
        for i, x in enumerate(lengths):
            if i > 2:
                x -= lengths[i - 2]
            more = True
            while more:
                c = x & 0x1F
                x >>= 5
                more = (x != 0) if ((c & 0x10) == 0) else (x != -1)
                if more:
                    c |= 0x20
                out.append(chr(c + 48))
        return "".join(out)

    def _decode_rle_pure(rle_obj: dict[str, Any]) -> np.ndarray:
        h, w = int(rle_obj["size"][0]), int(rle_obj["size"][1])
        s = rle_obj["counts"]
        if isinstance(s, (bytes, bytearray)):
            s = s.decode("ascii")
        cnts: list[int] = []
        p = 0
        m = 0
        while p < len(s):
            x = 0
            k = 0
            more = True
            while more:
                c = ord(s[p]) - 48
                x |= (c & 0x1F) << (5 * k)
                more = (c & 0x20) != 0
                p += 1
                k += 1
                if not more and (c & 0x10) != 0:
                    x |= -1 << (5 * k)
            if m > 2:
                x += cnts[m - 2]
            cnts.append(x)
            m += 1
        flat = np.zeros(h * w, dtype=np.uint8)
        pos = 0
        val = 0
        for run in cnts:
            if val == 1 and run > 0:
                flat[pos : pos + run] = 1
            pos += run
            val ^= 1
        return flat.reshape((h, w), order="F")

    def _area_single(rle_obj: dict[str, Any]) -> int:
        s = rle_obj["counts"]
        if isinstance(s, (bytes, bytearray)):
            s = s.decode("ascii")
        cnts: list[int] = []
        p = 0
        m = 0
        while p < len(s):
            x = 0
            k = 0
            more = True
            while more:
                c = ord(s[p]) - 48
                x |= (c & 0x1F) << (5 * k)
                more = (c & 0x20) != 0
                p += 1
                k += 1
                if not more and (c & 0x10) != 0:
                    x |= -1 << (5 * k)
            if m > 2:
                x += cnts[m - 2]
            cnts.append(x)
            m += 1
        return int(sum(cnts[1::2]))

    class _PureMaskUtil:
        @staticmethod
        def encode(mask: np.ndarray) -> dict[str, Any]:
            m_f = np.asfortranarray(mask.astype(np.uint8))
            return {"size": [int(m_f.shape[0]), int(m_f.shape[1])], "counts": _encode_rle_pure(m_f).encode("ascii")}

        @staticmethod
        def decode(rle_obj: dict[str, Any]) -> np.ndarray:
            return _decode_rle_pure(rle_obj)

        @staticmethod
        def area(rle_or_list: dict[str, Any] | list[dict[str, Any]]) -> Any:
            if isinstance(rle_or_list, list):
                return np.array([_area_single(r) for r in rle_or_list], dtype=np.uint64)
            return _area_single(rle_or_list)

        @staticmethod
        def iou(dt: list[dict[str, Any]], gt: list[dict[str, Any]], iscrowd: list[int]) -> np.ndarray:
            nd, ng = len(dt), len(gt)
            out = np.zeros((nd, ng), dtype=np.float64)
            dm = [_decode_rle_pure(r) > 0 for r in dt]
            gm = [_decode_rle_pure(r) > 0 for r in gt]
            for i in range(nd):
                ai = float(dm[i].sum())
                for j in range(ng):
                    aj = float(gm[j].sum())
                    inter = float(np.logical_and(dm[i], gm[j]).sum())
                    if inter > 0.0:
                        denom = ai if (iscrowd[j] if j < len(iscrowd) else 0) else (ai + aj - inter)
                        out[i, j] = inter / max(1.0, denom)
            return out

    mu = _PureMaskUtil()


class _RefereeShim:
    @staticmethod
    def rle(counts: str) -> dict[str, Any]:
        return {"size": [2048, 2048], "counts": counts.encode() if isinstance(counts, str) else counts}


referee_metric = _RefereeShim()
print(f"Environment ready | torch={torch.__version__} | timm={timm.__version__} | cuda={torch.cuda.is_available()} | pycocotools={HAS_PYCOCOTOOLS}", flush=True)

In [ ]:
"""Cell 2: Solar GONG Physics v2 (limb circle fit, limb flattening, dark depth, ribbonness, 5-ch GPU tensor)."""
IMG_SIZE = 2048


SLICE_SIZE = 1888  # 59 * 32; encloses 100.0% of solar chromosphere (2 * R_sun <= 1844 < 1888) at 1:1 native lattice


DEFAULT_CX = 1021.5


DEFAULT_CY = 1022.2


DEFAULT_R = 906.8


def _algebraic_circle_irls(xs: np.ndarray, ys: np.ndarray, iters: int = 4, delta: float = 2.0) -> tuple[float, float, float, float]:
    """Fit circle (cx, cy, R) with Huber IRLS; returns (cx, cy, R, rmse)."""
    w = np.ones_like(xs, dtype=np.float64)
    A_mat = np.column_stack([xs, ys, np.ones_like(xs)])
    rhs = -(xs * xs + ys * ys)
    cx, cy, R = DEFAULT_CX, DEFAULT_CY, DEFAULT_R
    for _ in range(iters):
        sw = np.sqrt(w)[:, None]
        coef, *_ = np.linalg.lstsq(A_mat * sw, rhs * sw[:, 0], rcond=None)
        cx = -0.5 * float(coef[0])
        cy = -0.5 * float(coef[1])
        r2 = cx * cx + cy * cy - float(coef[2])
        if r2 <= 1.0:
            return DEFAULT_CX, DEFAULT_CY, DEFAULT_R, 99.0
        R = float(np.sqrt(r2))
        dists = np.hypot(xs - cx, ys - cy) - R
        abs_d = np.abs(dists)
        w = np.where(abs_d <= delta, 1.0, delta / np.maximum(abs_d, 1e-6))
    rmse = float(np.sqrt(np.mean((np.hypot(xs - cx, ys - cy) - R) ** 2)))
    return cx, cy, R, rmse


def fit_gong_limb(img_u8: np.ndarray) -> tuple[float, float, float, int]:
    """Locate the inner H-alpha solar limb (R in [892, 922] px), rejecting outer coronagraph edges."""
    ys_m, xs_m = np.nonzero(img_u8[::4, ::4] > 75)
    if len(xs_m) > 100:
        cx0 = float(np.clip(0.5 * (float(xs_m.mean()) * 4.0) + 0.5 * DEFAULT_CX, 995.0, 1050.0))
        cy0 = float(np.clip(0.5 * (float(ys_m.mean()) * 4.0) + 0.5 * DEFAULT_CY, 995.0, 1050.0))
    else:
        cx0, cy0 = DEFAULT_CX, DEFAULT_CY

    thetas = np.linspace(0.0, 2.0 * np.pi, 360, endpoint=False, dtype=np.float32)
    radii = np.arange(860.0, 946.0, 1.0, dtype=np.float32)
    cos_t = np.cos(thetas)[:, None]
    sin_t = np.sin(thetas)[:, None]
    img_f = img_u8.astype(np.float32)

    cx, cy, R, rmse = cx0, cy0, DEFAULT_R, 99.0
    n_inliers = 0
    for pass_idx in range(2):
        map_x = (cx + cos_t * radii[None, :]).astype(np.float32)
        map_y = (cy + sin_t * radii[None, :]).astype(np.float32)
        prof = cv2.remap(img_f, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
        prof_s = prof.copy()
        prof_s[:, 1:-1] = 0.25 * prof[:, :-2] + 0.5 * prof[:, 1:-1] + 0.25 * prof[:, 2:]

        r_lo = int(882 - 860)
        r_hi = int(926 - 860)
        grad = np.zeros_like(prof_s)
        grad[:, 4:-4] = prof_s[:, :-8] - prof_s[:, 8:]
        sub = grad[:, r_lo:r_hi]
        best_idx = np.argmax(sub, axis=1) + r_lo
        best_r = radii[best_idx]
        best_g = grad[np.arange(len(thetas)), best_idx]
        inner_i = prof_s[np.arange(len(thetas)), np.maximum(0, best_idx - 14)]

        valid = (inner_i > 58.0) & (best_g > 9.0) & (best_r > 883.0) & (best_r < 925.0)
        if pass_idx == 1 and 890.0 <= R <= 924.0:
            valid &= np.abs(best_r - R) <= 8.0
        n_inliers = int(valid.sum())
        if n_inliers < 45:
            return DEFAULT_CX, DEFAULT_CY, DEFAULT_R, n_inliers

        px = cx + cos_t[valid, 0] * best_r[valid]
        py = cy + sin_t[valid, 0] * best_r[valid]
        cx_fit, cy_fit, R_fit, rmse_fit = _algebraic_circle_irls(px.astype(np.float64), py.astype(np.float64))
        if 892.0 <= R_fit <= 922.0 and abs(cx_fit - DEFAULT_CX) <= 50.0 and abs(cy_fit - DEFAULT_CY) <= 50.0:
            cx, cy, R, rmse = cx_fit, cy_fit, R_fit, rmse_fit
        else:
            return DEFAULT_CX, DEFAULT_CY, DEFAULT_R, n_inliers

    if not (892.0 <= R <= 922.0) or rmse > 8.0:
        return DEFAULT_CX, DEFAULT_CY, DEFAULT_R, n_inliers
    return float(cx), float(cy), float(R), n_inliers


def _clamp_exterior_512(img512: np.ndarray, cx: float, cy: float, R_sun: float) -> tuple[np.ndarray, np.ndarray]:
    cx5 = cx / 4.0
    cy5 = cy / 4.0
    r_lim = 0.982 * (R_sun / 4.0)
    yy, xx = np.mgrid[0:512, 0:512].astype(np.float32)
    dx = xx - cx5
    dy = yy - cy5
    rad = np.hypot(dx, dy)
    scale = np.where(rad > r_lim, r_lim / np.maximum(rad, 1e-6), 1.0).astype(np.float32)
    mx = (cx5 + dx * scale).astype(np.float32)
    my = (cy5 + dy * scale).astype(np.float32)
    clamped = cv2.remap(img512, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    on_disk512 = rad <= (0.990 * (R_sun / 4.0))
    return clamped, on_disk512


def compute_solar_geometry(cx: float, cy: float, R_sun: float, size: int = IMG_SIZE) -> tuple[np.ndarray, np.ndarray]:
    """Return (on_disk bool mask, mu cosine heliocentric angle float32) at native resolution."""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    rho = np.hypot(xx - np.float32(cx), yy - np.float32(cy)) / np.float32(R_sun)
    on_disk = rho <= np.float32(0.996)
    mu = np.sqrt(np.clip(1.0 - rho * rho, 0.0, 1.0)).astype(np.float32)
    return on_disk, mu


def _dark_ridge_tubularity_1024(d_multi_u8: np.ndarray, on_disk_1024: np.ndarray) -> np.ndarray:
    d1024 = cv2.resize(d_multi_u8, (1024, 1024), interpolation=cv2.INTER_AREA).astype(np.float32) * (1.0 / 255.0)
    best = np.zeros((1024, 1024), dtype=np.float32)
    for sigma in (1.2, 2.4, 4.8):
        k = max(3, int(round(sigma * 6.0)) | 1)
        sm = cv2.GaussianBlur(d1024, (k, k), sigmaX=sigma, sigmaY=sigma)
        dxx = (np.roll(sm, -1, axis=1) - 2.0 * sm + np.roll(sm, 1, axis=1)) * (sigma * sigma)
        dyy = (np.roll(sm, -1, axis=0) - 2.0 * sm + np.roll(sm, 1, axis=0)) * (sigma * sigma)
        dxy = 0.25 * (
            np.roll(np.roll(sm, -1, axis=0), -1, axis=1)
            - np.roll(np.roll(sm, -1, axis=0), 1, axis=1)
            - np.roll(np.roll(sm, 1, axis=0), -1, axis=1)
            + np.roll(np.roll(sm, 1, axis=0), 1, axis=1)
        ) * (sigma * sigma)
        tr = -(dxx + dyy)
        det_diff = np.sqrt(np.maximum(0.0, (dxx - dyy) ** 2 + 4.0 * dxy * dxy))
        lam_pos = 0.5 * (tr + det_diff)
        resp = np.maximum(0.0, lam_pos) * np.clip(d1024 * 4.0, 0.25, 1.0)
        best = np.maximum(best, resp)

    vals = best[on_disk_1024]
    scale = float(np.percentile(vals, 99.5)) if vals.size > 100 else 0.05
    scale = max(scale, 0.04)
    norm = np.clip(best / scale, 0.0, 1.0)
    norm[~on_disk_1024] = 0.0
    return (norm * 255.0).round().astype(np.uint8)


def build_physics_channels_u8(
    img_u8: np.ndarray,
    cx: float | None = None,
    cy: float | None = None,
    R_sun: float | None = None,
) -> tuple[np.ndarray, dict[str, float]]:
    """Build 2048x2048x3 uint8 solar physics tensor [I_flat, D_multi, T_ribbon]."""
    if cx is None or cy is None or R_sun is None:
        cx, cy, R_sun, n_inl = fit_gong_limb(img_u8)
    else:
        n_inl = 360

    img_f32 = img_u8.astype(np.float32)
    img512 = cv2.resize(img_f32, (512, 512), interpolation=cv2.INTER_AREA)
    clamped512, _ = _clamp_exterior_512(img512, cx, cy, R_sun)

    m0 = cv2.GaussianBlur(clamped512, (0, 0), 18.0)
    no_plage = np.minimum(clamped512, m0 + 18.0)
    k21 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    b_close = cv2.morphologyEx(no_plage, cv2.MORPH_CLOSE, k21)
    b_mid512 = cv2.GaussianBlur(b_close, (0, 0), 7.0)
    b_broad512 = cv2.GaussianBlur(b_close, (0, 0), 20.0)
    s_2d512 = np.maximum(cv2.GaussianBlur(np.abs(no_plage - b_mid512) * 1.4826, (0, 0), 16.0), 3.5)

    b_mid = cv2.resize(b_mid512, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    b_broad = cv2.resize(b_broad512, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    s_2d = cv2.resize(s_2d512, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

    on_disk, _ = compute_solar_geometry(cx, cy, R_sun, IMG_SIZE)
    on_disk_1024 = cv2.resize(on_disk.astype(np.uint8), (1024, 1024), interpolation=cv2.INTER_NEAREST) > 0

    diff_mid = b_mid - img_f32
    diff_broad = b_broad - img_f32

    ch0_f = np.clip(185.0 - 28.0 * (diff_mid / s_2d), 0.0, 255.0)
    ch0_u8 = np.where(on_disk, ch0_f, 185.0).round().astype(np.uint8)

    d_mid = np.clip(diff_mid / (6.5 * s_2d), 0.0, 1.0)
    d_broad = np.clip(diff_broad / (7.5 * s_2d), 0.0, 1.0)
    d_multi_u8 = np.where(on_disk, (0.55 * d_mid + 0.45 * d_broad) * 255.0, 0.0).round().astype(np.uint8)

    t_1024 = _dark_ridge_tubularity_1024(d_multi_u8, on_disk_1024)
    ch2_u8 = cv2.resize(t_1024, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    ch2_u8[~on_disk] = 0

    hw3 = np.stack([ch0_u8, d_multi_u8, ch2_u8], axis=-1)
    geom = {
        "cx": float(cx),
        "cy": float(cy),
        "R_sun": float(R_sun),
        "n_inliers": int(n_inl),
        "s_2d_med": float(np.median(s_2d[on_disk])),
    }
    return hw3, geom


def active_disk_slice_origin(
    cx: float,
    cy: float,
    slice_sz: int = SLICE_SIZE,
    img_size: int = IMG_SIZE,
) -> tuple[int, int]:
    """Integer top-left (x0, y0) of the 1:1 native slice enclosing the full solar disk."""
    max_off = max(0, img_size - slice_sz)
    x0 = int(np.clip(round(float(cx) - 0.5 * slice_sz), 0, max_off))
    y0 = int(np.clip(round(float(cy) - 0.5 * slice_sz), 0, max_off))
    return x0, y0


def build_native_5ch_gpu(
    hw3_bchw_u8: torch.Tensor,
    local_cxcy_r: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute 5-channel ImageNet-polarity native physics tensor on GPU.

    Args:
      hw3_bchw_u8: (B, 3, H, W) uint8/float tensor containing [I_flat, D_multi, T_ribbon].
      local_cxcy_r: (B, 3) float32 tensor containing (cx_local, cy_local, R_sun) in crop coords.
    Returns:
      x_5ch: (B, 5, H, W) float32 in [0, 1]: [I_flat, 1 - D_multi, 1 - D_barb15, T_ribbon, mu_disk].
      on_disk: (B, H, W) float32 in {0.0, 1.0}.
    """
    dev = hw3_bchw_u8.device
    b, _, h, w = hw3_bchw_u8.shape
    f32 = hw3_bchw_u8.float() * (1.0 / 255.0)
    i_flat = f32[:, 0:1]
    d_multi = f32[:, 1:2]
    t_ribbon = f32[:, 2:3]

    gx = torch.arange(w, device=dev, dtype=torch.float32).view(1, 1, 1, w)
    gy = torch.arange(h, device=dev, dtype=torch.float32).view(1, 1, h, 1)
    cx = local_cxcy_r[:, 0].to(dtype=torch.float32, device=dev).view(b, 1, 1, 1)
    cy = local_cxcy_r[:, 1].to(dtype=torch.float32, device=dev).view(b, 1, 1, 1)
    r_sun = local_cxcy_r[:, 2].to(dtype=torch.float32, device=dev).view(b, 1, 1, 1).clamp(min=100.0)

    rho2 = ((gx - cx) ** 2 + (gy - cy) ** 2) / (r_sun * r_sun)
    on_disk = (rho2 <= 0.996).to(torch.float32)
    mu_disk = torch.sqrt(torch.clamp(1.0 - rho2, min=0.0, max=1.0)) * on_disk

    # Native 1:1 15x15 fine barb local contrast enhancement (no downsampling!)
    bg_15 = F.avg_pool2d(i_flat, kernel_size=15, stride=1, padding=7, count_include_pad=False)
    d_barb = torch.clamp((bg_15 - i_flat) * 10.0, min=0.0, max=1.0) * on_disk

    # Polarity aligned with ImageNet on ch 0..2: bright quiet Sun, dark filaments
    x_5ch = torch.cat([i_flat, (1.0 - d_multi), (1.0 - d_barb), t_ribbon, mu_disk], dim=1)
    return x_5ch, on_disk[:, 0]

In [ ]:
"""Cell 3: Stage-1 Complementary 1408px Mask R-CNN v2 Detector & Multi-Fold Proposal Clustering."""
DET_SIZE = 1408


@dataclass
class ProposalROI:
    bbox: tuple[int, int, int, int]  # (x1, y1, x2, y2) in native 2048x2048
    mask_f16: np.ndarray             # shape (y2-y1, x2-x1), float16 in [0, 1]
    s_det: float
    view_agree: float


def compute_active_disk_box(geom: dict[str, float]) -> tuple[int, int, int]:
    """Return (x0, y0, side) enclosing the solar chromosphere with 14 px margin."""
    cx, cy, R_sun = geom["cx"], geom["cy"], geom["R_sun"]
    half = R_sun + 14.0
    x0 = int(round(cx - half))
    y0 = int(round(cy - half))
    side = int(round(2.0 * half))
    return x0, y0, max(512, side)


def _crop_and_resize_disk(hw3: np.ndarray, x0: int, y0: int, side: int, det_size: int = DET_SIZE) -> np.ndarray:
    canvas = np.zeros((side, side, 3), dtype=np.uint8)
    canvas[..., 0] = 185
    sx1, sy1 = max(0, x0), max(0, y0)
    sx2, sy2 = min(IMG_SIZE, x0 + side), min(IMG_SIZE, y0 + side)
    if sx2 > sx1 and sy2 > sy1:
        dx1, dy1 = sx1 - x0, sy1 - y0
        dx2, dy2 = dx1 + (sx2 - sx1), dy1 + (sy2 - sy1)
        canvas[dy1:dy2, dx1:dx2] = hw3[sy1:sy2, sx1:sx2]
    return cv2.resize(canvas, (det_size, det_size), interpolation=cv2.INTER_LINEAR)


def _install_ignore_shield(roi_heads: nn.Module) -> None:
    def _shielded_select(self, proposals, targets):
        self.check_targets(targets)
        assert targets is not None
        dtype = proposals[0].dtype
        device = proposals[0].device

        gt_boxes = [t["boxes"].to(dtype) for t in targets]
        gt_labels = [t["labels"] for t in targets]
        ign_boxes = [t.get("ignore_boxes") for t in targets]

        proposals = self.add_gt_proposals(proposals, gt_boxes)
        matched_idxs, labels = self.assign_targets_to_proposals(proposals, gt_boxes, gt_labels)

        for img_id in range(len(proposals)):
            ib = ign_boxes[img_id]
            if ib is not None and ib.numel() > 0:
                bg_mask = labels[img_id] == 0
                if bg_mask.any():
                    bg_idx = torch.nonzero(bg_mask, as_tuple=False).squeeze(1)
                    bg_props = proposals[img_id][bg_idx]
                    ious = box_ops.box_iou(bg_props, ib.to(dtype=dtype, device=device))
                    max_iou = ious.max(dim=1).values
                    cx = 0.5 * (bg_props[:, 0] + bg_props[:, 2])[:, None]
                    cy = 0.5 * (bg_props[:, 1] + bg_props[:, 3])[:, None]
                    inside = (
                        (cx >= ib[None, :, 0].to(device))
                        & (cx <= ib[None, :, 2].to(device))
                        & (cy >= ib[None, :, 1].to(device))
                        & (cy <= ib[None, :, 3].to(device))
                    ).any(dim=1)
                    hit = (max_iou >= 0.25) | inside
                    if hit.any():
                        labels[img_id][bg_idx[hit]] = -1

        sampled_inds = self.subsample(labels)
        matched_gt_boxes = []
        for img_id in range(len(proposals)):
            inds = sampled_inds[img_id]
            proposals[img_id] = proposals[img_id][inds]
            labels[img_id] = labels[img_id][inds]
            matched_idxs[img_id] = matched_idxs[img_id][inds]
            gb = gt_boxes[img_id]
            if gb.numel() == 0:
                gb = torch.zeros((1, 4), dtype=dtype, device=device)
            matched_gt_boxes.append(gb[matched_idxs[img_id]])

        regression_targets = self.box_coder.encode(matched_gt_boxes, proposals)
        return proposals, matched_idxs, labels, regression_targets

    roi_heads.select_training_samples = types.MethodType(_shielded_select, roi_heads)


def build_stage1_model(pretrained: bool = True, det_size: int = DET_SIZE) -> nn.Module:
    weights = "DEFAULT" if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(
        weights=weights,
        weights_backbone=None,
        min_size=det_size,
        max_size=det_size,
        image_mean=[0.725, 0.045, 0.030],
        image_std=[0.180, 0.115, 0.095],
        rpn_post_nms_top_n_train=1000,
        rpn_post_nms_top_n_test=800,
        box_detections_per_img=140,
        box_score_thresh=0.05,
        box_nms_thresh=0.55,
    )
    ratios = (0.15, 0.33, 0.6, 1.0, 1.67, 3.0, 6.5)
    anchor_sizes = ((12,), (24,), (48,), (96,), (192,))
    model.rpn.anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=(ratios,) * 5)

    conv_depth = len(model.rpn.head.conv) if hasattr(model.rpn.head, "conv") else 2
    old_conv_state = copy.deepcopy(model.rpn.head.conv.state_dict()) if hasattr(model.rpn.head, "conv") else None
    model.rpn.head = RPNHead(model.backbone.out_channels, len(ratios), conv_depth=conv_depth)
    if old_conv_state is not None:
        model.rpn.head.conv.load_state_dict(old_conv_state)

    model.roi_heads.mask_roi_pool.output_size = (28, 28)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, 2)
    _install_ignore_shield(model.roi_heads)
    return model


def load_stage1_detector_from_fp16(ckpt_path: Path, device: torch.device, det_size: int = DET_SIZE) -> nn.Module:
    model = build_stage1_model(pretrained=False, det_size=det_size).to(device)
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd32 = {k: (v.float() if torch.is_tensor(v) and v.is_floating_point() else v) for k, v in sd.items()}
    model.load_state_dict(sd32)
    model.to(device).eval()
    return model


def _view_transform(img_t: torch.Tensor, v: int) -> torch.Tensor:
    if v == 1:
        return torch.flip(img_t, dims=[2])
    if v == 2:
        return torch.flip(img_t, dims=[1])
    if v == 3:
        return torch.flip(img_t, dims=[1, 2])
    return img_t


def _invert_box_coords(b: np.ndarray, v: int, det_size: int) -> np.ndarray:
    x1, y1, x2, y2 = float(b[0]), float(b[1]), float(b[2]), float(b[3])
    if v in (1, 3):
        x1, x2 = float(det_size) - x2, float(det_size) - x1
    if v in (2, 3):
        y1, y2 = float(det_size) - y2, float(det_size) - y1
    return np.array([x1, y1, x2, y2], dtype=np.float32)


def _invert_crop_gpu(crop_hw: torch.Tensor, v: int) -> torch.Tensor:
    out = crop_hw
    if v in (1, 3):
        out = torch.flip(out, dims=[1])
    if v in (2, 3):
        out = torch.flip(out, dims=[0])
    return out


def cluster_multiview_proposals(
    candidates: list[dict[str, Any]],
    iou_thr: float = 0.45,
    n_views: int = 4,
    max_keep: int = 120,
) -> list[ProposalROI]:
    """Cluster view-tagged Mask R-CNN proposals across TTA views (and/or folds)."""
    if not candidates:
        return []
    candidates = sorted(candidates, key=lambda c: -c["score"])
    used = [False] * len(candidates)
    out: list[ProposalROI] = []

    for i, c0 in enumerate(candidates):
        if used[i]:
            continue
        used[i] = True
        cluster = [c0]
        bx1, by1, bx2, by2 = c0["bbox"]
        a1 = float((c0["mask"] > 0.35).sum())
        for j in range(i + 1, len(candidates)):
            if used[j]:
                continue
            cj = candidates[j]
            jx1, jy1, jx2, jy2 = cj["bbox"]
            ix1, iy1 = max(bx1, jx1), max(by1, jy1)
            ix2, iy2 = min(bx2, jx2), min(by2, jy2)
            if ix2 <= ix1 or iy2 <= iy1:
                continue
            m1_sub = c0["mask"][iy1 - by1 : iy2 - by1, ix1 - bx1 : ix2 - bx1] > 0.35
            m2_sub = cj["mask"][iy1 - jy1 : iy2 - jy1, ix1 - jx1 : ix2 - jx1] > 0.35
            inter = float(np.logical_and(m1_sub, m2_sub).sum())
            if inter <= 0.0:
                continue
            a2 = float((cj["mask"] > 0.35).sum())
            iou = inter / max(1.0, a1 + a2 - inter)
            if iou >= iou_thr:
                used[j] = True
                cluster.append(cj)

        ux1 = min(c["bbox"][0] for c in cluster)
        uy1 = min(c["bbox"][1] for c in cluster)
        ux2 = max(c["bbox"][2] for c in cluster)
        uy2 = max(c["bbox"][3] for c in cluster)
        acc = np.zeros((uy2 - uy1, ux2 - ux1), dtype=np.float32)
        w_sum = 0.0
        best_by_view: dict[int, float] = {}
        for c in cluster:
            cx1, cy1, cx2, cy2 = c["bbox"]
            sc = float(c["score"])
            acc[cy1 - uy1 : cy2 - uy1, cx1 - ux1 : cx2 - ux1] += sc * c["mask"].astype(np.float32)
            w_sum += sc
            v = int(c["view"])
            if sc > best_by_view.get(v, 0.0):
                best_by_view[v] = sc
        fused = (acc / max(1e-6, w_sum)).astype(np.float16)
        ys, xs = np.nonzero(fused > 0.28)
        if xs.size < 16:
            continue
        tx1, ty1 = int(xs.min()), int(ys.min())
        tx2, ty2 = int(xs.max()) + 1, int(ys.max()) + 1
        s_det = sum(best_by_view.values()) / float(max(1, n_views))
        v_agree = float(len(best_by_view)) / float(max(1, n_views))
        if s_det < 0.08:
            continue
        out.append(
            ProposalROI(
                bbox=(ux1 + tx1, uy1 + ty1, ux1 + tx2, uy1 + ty2),
                mask_f16=fused[ty1:ty2, tx1:tx2].copy(),
                s_det=float(s_det),
                view_agree=float(v_agree),
            )
        )
        if len(out) >= max_keep:
            break
    out.sort(key=lambda p: -p.s_det)
    return out


@torch.inference_mode()
def predict_stage1_raw_candidates(
    model: nn.Module,
    stems: list[str],
    cache_dir: Path,
    geom_map: dict[str, dict[str, float]],
    device: torch.device,
    ram_physics: dict[str, np.ndarray] | None = None,
    det_size: int = DET_SIZE,
    tta_views: tuple[int, ...] = (0, 1, 2, 3),
    score_thresh: float = 0.10,
    view_offset: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Return unclustered view-tagged raw Mask R-CNN candidates (native 2048 coords)."""
    model.eval()
    use_bf16 = (
        device.type == "cuda"
        and torch.cuda.get_device_capability(device)[0] >= 8
        and torch.cuda.is_bf16_supported()
    )
    use_amp = device.type == "cuda"
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    results: dict[str, list[dict[str, Any]]] = {}

    for stem in stems:
        hw3 = ram_physics[stem] if (ram_physics and stem in ram_physics) else load_cached_physics_u8(cache_dir, stem)
        geom = geom_map[stem]
        x0, y0, side = compute_active_disk_box(geom)
        scale = float(det_size) / float(side)
        img_det = _crop_and_resize_disk(hw3, x0, y0, side, det_size)
        base_t = torch.from_numpy(img_det.transpose(2, 0, 1).copy()).float() * (1.0 / 255.0)
        base_t = base_t.to(device)

        views_tensors = [_view_transform(base_t, v) for v in tta_views]
        cands: list[dict[str, Any]] = []
        for start in range(0, len(views_tensors), 2):
            sub_v = tta_views[start : start + 2]
            sub_t = views_tensors[start : start + 2]
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                preds = model(sub_t)
            for v, pred in zip(sub_v, preds):
                scores_t = pred["scores"].float()
                keep_idx = torch.nonzero(scores_t >= score_thresh, as_tuple=False).squeeze(1)
                if keep_idx.numel() == 0:
                    continue
                boxes_np = pred["boxes"][keep_idx].float().cpu().numpy()
                scores_np = scores_t[keep_idx].cpu().numpy()
                masks_t = pred["masks"][keep_idx, 0]

                for k_i in range(len(scores_np)):
                    bx1, by1, bx2, by2 = boxes_np[k_i]
                    vx1 = max(0, int(np.floor(bx1)) - 2)
                    vy1 = max(0, int(np.floor(by1)) - 2)
                    vx2 = min(det_size, int(np.ceil(bx2)) + 2)
                    vy2 = min(det_size, int(np.ceil(by2)) + 2)
                    if vx2 <= vx1 + 1 or vy2 <= vy1 + 1:
                        continue
                    sub_gpu = masks_t[k_i, vy1:vy2, vx1:vx2].float()
                    if float(sub_gpu.max()) < 0.25:
                        continue
                    sub_gpu = _invert_crop_gpu(sub_gpu, v)

                    ux1, uy1, ux2, uy2 = _invert_box_coords(np.array([vx1, vy1, vx2, vy2]), v, det_size)
                    nx1 = int(np.clip(np.floor(x0 + ux1 / scale), 0, IMG_SIZE - 1))
                    ny1 = int(np.clip(np.floor(y0 + uy1 / scale), 0, IMG_SIZE - 1))
                    nx2 = int(np.clip(np.ceil(x0 + ux2 / scale), nx1 + 2, IMG_SIZE))
                    ny2 = int(np.clip(np.ceil(y0 + uy2 / scale), ny1 + 2, IMG_SIZE))
                    pcx, pcy = 0.5 * (nx1 + nx2), 0.5 * (ny1 + ny2)
                    if math.hypot(pcx - geom["cx"], pcy - geom["cy"]) > 0.996 * geom["R_sun"]:
                        continue

                    nat_t = F.interpolate(
                        sub_gpu.unsqueeze(0).unsqueeze(0),
                        size=(ny2 - ny1, nx2 - nx1),
                        mode="bilinear",
                        align_corners=False,
                    )[0, 0]
                    cands.append({
                        "view": view_offset + int(v),
                        "score": float(scores_np[k_i]),
                        "bbox": (nx1, ny1, nx2, ny2),
                        "mask": nat_t.to(torch.float16).cpu().numpy(),
                    })
        results[stem] = cands
    return results

def load_cached_physics_u8(cache_dir: Path, stem: str) -> np.ndarray:  # pragma: no cover
    raise RuntimeError("Streaming notebook expects in-memory ram_physics")


@torch.inference_mode()
def predict_stage1_single(
    model: nn.Module,
    hw3_u8: np.ndarray,
    geom: dict[str, float],
    device: torch.device,
    det_size: int = DET_SIZE,
    tta_views: tuple[int, ...] = (0, 1, 2, 3),
    score_thresh: float = 0.10,
    view_offset: int = 0,
) -> list[dict[str, Any]]:
    """Run Stage-1 TTA inference directly on an in-memory physics tensor."""
    res = predict_stage1_raw_candidates(
        model=model,
        stems=["_cur_"],
        cache_dir=Path("."),
        geom_map={"_cur_": geom},
        device=device,
        ram_physics={"_cur_": hw3_u8},
        det_size=det_size,
        tta_views=tta_views,
        score_thresh=score_thresh,
        view_offset=view_offset,
    )
    return res.get("_cur_", [])

In [ ]:
"""Cell 4: Native 1:1 Heterogeneous Cooperative Co-DeepNet (ConvNeXt-Small + StripBarbUNet + CrossBridge)."""
def _gn(ch: int) -> nn.GroupNorm:
    for g in (16, 8, 4, 2, 1):
        if ch % g == 0:
            return nn.GroupNorm(g, ch)
    return nn.GroupNorm(1, ch)


class SCSEBlock(nn.Module):
    """Concurrent spatial + channel squeeze-and-excitation."""

    def __init__(self, ch: int, r: int = 8) -> None:
        super().__init__()
        mid = max(4, ch // r)
        self.c_se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, mid, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid, ch, 1),
            nn.Sigmoid(),
        )
        self.s_se = nn.Sequential(nn.Conv2d(ch, 1, 1), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.c_se(x) + x * self.s_se(x)


class ConvGNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, stride: int = 1, dil: int = 1) -> None:
        super().__init__()
        pad = ((k - 1) // 2) * dil
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=stride, padding=pad, dilation=dil, bias=False),
            _gn(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DecBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.c1 = ConvGNAct(in_ch, out_ch, 3)
        self.c2 = ConvGNAct(out_ch, out_ch, 3)
        self.scse = SCSEBlock(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.scse(self.c2(self.c1(x)))


class RibbonBarbBlock(nn.Module):
    """Multi-orientation anisotropic conv block (3x3 + 1x9 + 9x1 + 3x3 dilated) for fine barbs & thin ribbons."""

    def __init__(self, in_ch: int, out_ch: int, dil: int = 2) -> None:
        super().__init__()
        b = max(4, out_ch // 4)
        cat_ch = b * 4
        self.proj = nn.Conv2d(in_ch, b, kernel_size=1, bias=False)
        self.br_iso = nn.Conv2d(b, b, kernel_size=3, padding=1, bias=False)
        self.br_h = nn.Conv2d(b, b, kernel_size=(1, 9), padding=(0, 4), bias=False)
        self.br_v = nn.Conv2d(b, b, kernel_size=(9, 1), padding=(4, 0), bias=False)
        self.br_dil = nn.Conv2d(b, b, kernel_size=3, padding=dil, dilation=dil, bias=False)
        self.norm_act = nn.Sequential(_gn(cat_ch), nn.SiLU(inplace=True))
        self.out_conv = ConvGNAct(cat_ch, out_ch, 3)
        self.scse = SCSEBlock(out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.proj(x)
        h = torch.cat([self.br_iso(z), self.br_h(z), self.br_v(z), self.br_dil(z)], dim=1)
        h = self.scse(self.out_conv(self.norm_act(h)))
        return h + self.skip(x)


class CrossBridge(nn.Module):
    """Bidirectional zero-init gated feature exchange between Peer A (ConvNeXt) and Peer B (RibbonBarbUNet)."""

    def __init__(self, ch_a: int, ch_b: int) -> None:
        super().__init__()
        self.b2a = nn.Sequential(ConvGNAct(ch_b, ch_a, 1), nn.Conv2d(ch_a, ch_a, 1))
        self.a2b = nn.Sequential(ConvGNAct(ch_a, ch_b, 1), nn.Conv2d(ch_b, ch_b, 1))
        nn.init.zeros_(self.b2a[-1].weight)
        nn.init.zeros_(self.b2a[-1].bias)
        nn.init.zeros_(self.a2b[-1].weight)
        nn.init.zeros_(self.a2b[-1].bias)
        self.alpha_b2a = nn.Parameter(torch.tensor(0.0))
        self.alpha_a2b = nn.Parameter(torch.tensor(0.0))

    def forward(self, fa: torch.Tensor, fb: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        ga = torch.tanh(self.alpha_b2a)
        gb = torch.tanh(self.alpha_a2b)
        return fa + ga * self.b2a(fb), fb + gb * self.a2b(fa)


class NativeCoDeepNet(nn.Module):
    """Full-resolution 1:1 cooperative dual-network taking strictly 5 physical H-alpha channels (zero shortcut leakage)."""

    def __init__(self, in_ch: int = 5, pretrained: bool = True) -> None:
        super().__init__()
        try:
            self.backbone_a = timm.create_model(
                "convnext_small.fb_in22k_ft_in1k_384",
                pretrained=pretrained,
                features_only=True,
                out_indices=(0, 1, 2, 3),
            )
        except Exception:
            self.backbone_a = timm.create_model(
                "convnext_small",
                pretrained=False,
                features_only=True,
                out_indices=(0, 1, 2, 3),
            )
        assert hasattr(self.backbone_a, "stem_0"), "Expected timm ConvNeXt features_only stem_0"
        old_conv = self.backbone_a.stem_0
        assert isinstance(old_conv, nn.Conv2d) and old_conv.in_channels == 3
        new_conv = nn.Conv2d(
            in_ch,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None),
        )
        with torch.no_grad():
            new_conv.weight[:, :3].copy_(old_conv.weight)
            nn.init.normal_(new_conv.weight[:, 3:], mean=0.0, std=0.005)
            if old_conv.bias is not None and new_conv.bias is not None:
                new_conv.bias.copy_(old_conv.bias)
        self.backbone_a.stem_0 = new_conv

        self.register_buffer("im_mean", torch.tensor([0.485, 0.456, 0.406, 0.0, 0.0]).view(1, 5, 1, 1))
        self.register_buffer("im_std", torch.tensor([0.229, 0.224, 0.225, 1.0, 1.0]).view(1, 5, 1, 1))

        self.s1_stem_a = nn.Sequential(ConvGNAct(in_ch, 24, 3), ConvGNAct(24, 24, 3))
        self.s2_stem_a = nn.Sequential(ConvGNAct(24, 48, 3, stride=2), ConvGNAct(48, 48, 3))

        self.up16_a = DecBlock(768 + 384, 256)
        self.up8_a = DecBlock(256 + 192, 160)
        self.up4_a = DecBlock(160 + 96, 96)
        self.up2_a = DecBlock(96 + 48, 64)
        self.up1_a = DecBlock(64 + 24, 32)

        self.enc1_b = RibbonBarbBlock(in_ch, 24, dil=2)
        self.enc2_b = nn.Sequential(nn.AvgPool2d(2), RibbonBarbBlock(24, 48, dil=2))
        self.enc4_b = nn.Sequential(nn.AvgPool2d(2), RibbonBarbBlock(48, 96, dil=2))
        self.enc8_b = nn.Sequential(nn.AvgPool2d(2), RibbonBarbBlock(96, 144, dil=2))
        self.enc16_b = nn.Sequential(nn.AvgPool2d(2), RibbonBarbBlock(144, 192, dil=3))

        self.dec8_b = RibbonBarbBlock(192 + 144, 144, dil=2)
        self.dec4_b = RibbonBarbBlock(144 + 96, 96, dil=2)
        self.dec2_b = RibbonBarbBlock(96 + 48, 48, dil=2)
        self.dec1_b = RibbonBarbBlock(48 + 24, 24, dil=2)

        self.bridge16 = CrossBridge(256, 192)
        self.bridge8 = CrossBridge(160, 144)
        self.bridge4 = CrossBridge(96, 96)
        self.bridge2 = CrossBridge(64, 48)

        self.head_a = nn.Conv2d(32, 3, 1)
        self.head_b = nn.Conv2d(24, 3, 1)
        self.fuse_dec = nn.Sequential(
            ConvGNAct(32 + 24 + in_ch, 32, 3),
            ConvGNAct(32, 32, 3),
        )
        self.head_f = nn.Conv2d(32, 3, 1)

        bias_vec = torch.tensor([-4.6, -5.0, -5.0])
        for hd in (self.head_a, self.head_b, self.head_f):
            nn.init.normal_(hd.weight, mean=0.0, std=0.02)
            with torch.no_grad():
                hd.bias.copy_(bias_vec)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        xn = (x - self.im_mean) / self.im_std

        c0, c1, c2, c3 = self.backbone_a(xn)
        a_s1 = self.s1_stem_a(xn)
        a_s2 = self.s2_stem_a(a_s1)

        b_s1 = self.enc1_b(xn)
        b_s2 = self.enc2_b(b_s1)
        b_s4 = self.enc4_b(b_s2)
        b_s8 = self.enc8_b(b_s4)
        b_s16 = self.enc16_b(b_s8)

        d16_a = self.up16_a(torch.cat([F.interpolate(c3, size=c2.shape[-2:], mode="bilinear", align_corners=False), c2], dim=1))
        d16_a, d16_b = self.bridge16(d16_a, b_s16)

        d8_a = self.up8_a(torch.cat([F.interpolate(d16_a, size=c1.shape[-2:], mode="bilinear", align_corners=False), c1], dim=1))
        d8_b = self.dec8_b(torch.cat([F.interpolate(d16_b, size=b_s8.shape[-2:], mode="bilinear", align_corners=False), b_s8], dim=1))
        d8_a, d8_b = self.bridge8(d8_a, d8_b)

        d4_a = self.up4_a(torch.cat([F.interpolate(d8_a, size=c0.shape[-2:], mode="bilinear", align_corners=False), c0], dim=1))
        d4_b = self.dec4_b(torch.cat([F.interpolate(d8_b, size=b_s4.shape[-2:], mode="bilinear", align_corners=False), b_s4], dim=1))
        d4_a, d4_b = self.bridge4(d4_a, d4_b)

        d2_a = self.up2_a(torch.cat([F.interpolate(d4_a, size=a_s2.shape[-2:], mode="bilinear", align_corners=False), a_s2], dim=1))
        d2_b = self.dec2_b(torch.cat([F.interpolate(d4_b, size=b_s2.shape[-2:], mode="bilinear", align_corners=False), b_s2], dim=1))
        d2_a, d2_b = self.bridge2(d2_a, d2_b)

        fa = self.up1_a(torch.cat([F.interpolate(d2_a, size=a_s1.shape[-2:], mode="bilinear", align_corners=False), a_s1], dim=1))
        fb = self.dec1_b(torch.cat([F.interpolate(d2_b, size=b_s1.shape[-2:], mode="bilinear", align_corners=False), b_s1], dim=1))

        la = self.head_a(fa)
        lb = self.head_b(fb)
        ff = self.fuse_dec(torch.cat([fa, fb, x], dim=1))
        lf = self.head_f(ff)
        return lf, la, lb


def load_native_codeepnet_from_fp16(ckpt_path: Path, device: torch.device) -> nn.Module:
    """Instantiate NativeCoDeepNet strictly offline (pretrained=False) and restore fp16 weights in float32."""
    model = NativeCoDeepNet(in_ch=5, pretrained=False).to(device)
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd32 = {k: (v.float() if torch.is_tensor(v) and v.is_floating_point() else v) for k, v in sd.items()}
    model.load_state_dict(sd32)
    model.to(device).eval()
    return model


@torch.inference_mode()
def predict_native_full_disk(
    model: nn.Module,
    hw3_2048_u8: np.ndarray,
    geom: dict[str, float],
    device: torch.device,
    slice_size: int = SLICE_SIZE,
    tta_views: tuple[int, ...] = (0, 1, 2, 3),
) -> tuple[int, int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Run 1:1 pure integer slice 1888x1888 4x flip TTA inference with fp16-safe logit clamping."""
    model.eval()
    cx, cy, r_sun = float(geom["cx"]), float(geom["cy"]), float(geom["R_sun"])
    x0, y0 = active_disk_slice_origin(cx, cy, slice_sz=slice_size, img_size=IMG_SIZE)
    hw3_slice = np.ascontiguousarray(hw3_2048_u8[y0 : y0 + slice_size, x0 : x0 + slice_size].transpose(2, 0, 1))
    hw3_t = torch.from_numpy(hw3_slice).unsqueeze(0).to(device, non_blocking=True)
    meta_t = torch.tensor([[cx - float(x0), cy - float(y0), r_sun]], dtype=torch.float32, device=device)
    x_5ch, on_disk = build_native_5ch_gpu(hw3_t, meta_t)

    use_bf16 = (
        device.type == "cuda"
        and torch.cuda.get_device_capability(device)[0] >= 8
        and torch.cuda.is_bf16_supported()
    )
    use_amp = device.type == "cuda"
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

    acc_fg = torch.zeros((slice_size, slice_size), dtype=torch.float32, device=device)
    acc_core = torch.zeros_like(acc_fg)
    acc_spine = torch.zeros_like(acc_fg)
    acc_dis = torch.zeros_like(acc_fg)

    for v in tta_views:
        xv = x_5ch
        if v == 1:
            xv = torch.flip(x_5ch, dims=[3])
        elif v == 2:
            xv = torch.flip(x_5ch, dims=[2])
        elif v == 3:
            xv = torch.flip(x_5ch, dims=[2, 3])

        try:
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                lf, la, lb = model(xv)
        except RuntimeError:
            if device.type == "cuda":
                torch.cuda.empty_cache()
            try:
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    lf, la, lb = model(xv)
            except RuntimeError:
                lf, la, lb = model(xv.float())

        lf32 = torch.nan_to_num(lf[0].float(), nan=0.0, posinf=20.0, neginf=-20.0)
        la32 = torch.nan_to_num(la[0].float(), nan=0.0, posinf=20.0, neginf=-20.0)
        lb32 = torch.nan_to_num(lb[0].float(), nan=0.0, posinf=20.0, neginf=-20.0)

        pf = torch.sigmoid(lf32)
        pa = torch.sigmoid(la32)
        pb = torch.sigmoid(lb32)

        p3 = 0.50 * pf + 0.25 * pa + 0.25 * pb
        dis = torch.abs(pa[0] - pb[0])

        if v == 1:
            p3 = torch.flip(p3, dims=[2])
            dis = torch.flip(dis, dims=[1])
        elif v == 2:
            p3 = torch.flip(p3, dims=[1])
            dis = torch.flip(dis, dims=[0])
        elif v == 3:
            p3 = torch.flip(p3, dims=[1, 2])
            dis = torch.flip(dis, dims=[0, 1])

        acc_fg += p3[0]
        acc_core += p3[1]
        acc_spine += p3[2]
        acc_dis += dis

    inv_n = 1.0 / float(len(tta_views))
    od = on_disk[0]
    p_fg = (acc_fg * inv_n * od).cpu().numpy()
    p_core = (acc_core * inv_n * od).cpu().numpy()
    p_spine = (acc_spine * inv_n * od).cpu().numpy()
    p_dis = (acc_dis * inv_n * od).cpu().numpy()
    return x0, y0, p_fg, p_core, p_spine, p_dis

In [ ]:
"""Cell 5: Geodesic Hysteresis Candidate Extraction, 24-d Meta-Scorer, Zero-Overlap Composer & RLE Writer."""
_OWNER_BUF = np.full((IMG_SIZE, IMG_SIZE), -1, dtype=np.int32)


_SCORE_BUF = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)


_TMP_MASK = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8, order="F")


TAU_HYST_GRID: tuple[tuple[float, float], ...] = (
    (0.46, 0.00),
    (0.50, 0.00),
    (0.54, 0.00),
    (0.50, 0.06),
    (0.54, 0.06),
    (0.54, 0.10),
    (0.58, 0.08),
    (0.58, 0.12),
)


def tau_pair_key(tau_seed: float, delta_hyst: float) -> str:
    return f"{float(tau_seed):.2f}_{float(delta_hyst):.2f}"


FEATURE_NAMES = (
    "p_fg_mean",
    "p_fg_q85",
    "p_fg_max",
    "p_core_mean",
    "p_spine_cov",
    "peer_agree",
    "boundary_drop",
    "s_det_best",
    "det_iou_best",
    "s1_view_frac",
    "d_multi_med",
    "d_barb_med",
    "t_ribbon_med",
    "log_area",
    "thinness",
    "mu_cent",
    "p_fg_x_agree",
    "t_rib_q90",
    "d_multi_q90",
    "snr_local",
    "edge_contrast",
    "elongation",
    "circularity",
    "plage_ring_q95",
)


@dataclass
class HybridCandidate:
    bbox: tuple[int, int, int, int]      # (x1, y1, x2, y2) in native 2048x2048
    mask_u8: np.ndarray                  # (y2-y1, x2-x1) uint8 in {0, 1}
    features: np.ndarray                 # (24,) float32
    v_zero: float
    is_healed: bool = False
    is_rescued: bool = False
    v_hat: float = 0.0


def _mask_bbox_u8(m: np.ndarray) -> tuple[int, int, int, int] | None:
    ys, xs = np.nonzero(m)
    if xs.size == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def _find_closest_points(m1_pts: np.ndarray, m2_pts: np.ndarray) -> tuple[float, np.ndarray, np.ndarray]:
    """Find closest (y, x) pixel pair between two boundary point sets."""
    if len(m1_pts) > 96:
        m1_pts = m1_pts[np.linspace(0, len(m1_pts) - 1, 96, dtype=int)]
    if len(m2_pts) > 96:
        m2_pts = m2_pts[np.linspace(0, len(m2_pts) - 1, 96, dtype=int)]
    diff = m1_pts[:, None, :] - m2_pts[None, :, :]
    d2 = (diff * diff).sum(axis=2)
    flat = int(np.argmin(d2))
    i0, j0 = divmod(flat, d2.shape[1])
    return float(math.sqrt(float(d2[i0, j0]))), m1_pts[i0], m2_pts[j0]


def _contour_yx(mask_crop: np.ndarray, ox: int, oy: int) -> np.ndarray:
    b = mask_crop.astype(np.uint8)
    er = cv2.erode(b, np.ones((3, 3), dtype=np.uint8), iterations=1)
    edge = b & (~er)
    ys, xs = np.nonzero(edge if edge.any() else b)
    return np.stack([ys + oy, xs + ox], axis=1).astype(np.float32)


def extract_stem_candidates_multitau(
    stem: str,
    x0: int,
    y0: int,
    p_fg: np.ndarray,
    p_core: np.ndarray,
    p_spine: np.ndarray,
    p_disagree: np.ndarray,
    hw3_2048_u8: np.ndarray,
    geom: dict[str, float],
    s1_props: list[ProposalROI],
    tau_hyst_grid: tuple[tuple[float, float], ...] = TAU_HYST_GRID,
) -> dict[str, list[HybridCandidate]]:
    """Build crisp 1:1 native connected components via Geodesic Hysteresis + R-CNN touch splitting + waist healing."""
    sh, sw = p_fg.shape
    hw3_sl = np.asarray(hw3_2048_u8[y0 : y0 + sh, x0 : x0 + sw], dtype=np.uint8)
    i_flat_f = hw3_sl[..., 0].astype(np.float32)
    bg_15 = cv2.boxFilter(i_flat_f, -1, (15, 15), normalize=True)
    bg_sq_15 = cv2.boxFilter(i_flat_f * i_flat_f, -1, (15, 15), normalize=True)
    loc_sd_sl = np.sqrt(np.maximum(bg_sq_15 - bg_15 * bg_15, 4.0))
    plage_sl = np.clip((i_flat_f - bg_15) * (6.0 / 255.0), 0.0, 1.0)
    d_barb_sl = np.clip((bg_15 - i_flat_f) * (10.0 / 255.0), 0.0, 1.0)
    d_multi_sl = hw3_sl[..., 1].astype(np.float32) * (1.0 / 255.0)
    t_rib_sl = hw3_sl[..., 2].astype(np.float32) * (1.0 / 255.0)

    # Pre-project Stage-1 proposals into slice-local coordinates
    local_s1: list[dict[str, Any]] = []
    for pr in s1_props:
        bx1, by1, bx2, by2 = pr.bbox
        sx1 = max(0, bx1 - x0)
        sy1 = max(0, by1 - y0)
        sx2 = min(sw, bx2 - x0)
        sy2 = min(sh, by2 - y0)
        if sx2 <= sx1 or sy2 <= sy1:
            continue
        pm = pr.mask_f16[(sy1 + y0 - by1) : (sy2 + y0 - by1), (sx1 + x0 - bx1) : (sx2 + x0 - bx1)] >= 0.48
        area_d = int(pm.sum())
        if area_d < 16:
            continue
        local_s1.append({
            "sbbox": (sx1, sy1, sx2, sy2),
            "mask": pm,
            "area": area_d,
            "s_det": float(pr.s_det),
            "view_agree": float(pr.view_agree),
        })

    cx_s, cy_s, r_sun = float(geom["cx"]), float(geom["cy"]), max(100.0, float(geom["R_sun"]))
    k3 = np.ones((3, 3), dtype=np.uint8)
    k3_bool = np.ones((3, 3), dtype=bool)
    out_by_tau: dict[str, list[HybridCandidate]] = {}

    for tau_seed, delta_hyst in tau_hyst_grid:
        tkey = tau_pair_key(tau_seed, delta_hyst)
        spine_gate = max(0.28, float(tau_seed) - 0.12)
        m_seed = (p_fg >= tau_seed) | ((p_spine >= 0.48) & (p_fg >= spine_gate))
        if delta_hyst > 1e-6:
            low_fg = max(0.24, float(tau_seed) - float(delta_hyst))
            low_sp_fg = max(0.22, float(tau_seed) - float(delta_hyst) - 0.04)
            m_low = (p_fg >= low_fg) | ((p_spine >= 0.42) & (p_fg >= low_sp_fg))
            m_bin = ndimage.binary_propagation(m_seed, mask=(m_low | m_seed), structure=k3_bool)
        else:
            m_bin = m_seed
        n_cc, lbl_map, stats, _ = cv2.connectedComponentsWithStats(m_bin.astype(np.uint8), connectivity=8)

        # Raw components -> list of (sbbox, mask_crop, is_healed, is_rescued)
        raw_comps: list[tuple[tuple[int, int, int, int], np.ndarray, bool, bool]] = []
        for cid in range(1, n_cc):
            cx1, cy1, cw, ch, area_c = [int(v) for v in stats[cid]]
            if area_c < 20:
                continue
            cx2, cy2 = cx1 + cw, cy1 + ch
            cmask = (lbl_map[cy1:cy2, cx1:cx2] == cid)

            # 1. Detector-Verified Touch Separation (only when >=2 disjoint R-CNN objects overlap >=18% each)
            split_done = False
            if area_c >= 140 and len(local_s1) >= 2:
                overlaps = []
                for di, det in enumerate(local_s1):
                    if det["s_det"] < 0.44:
                        continue
                    dx1, dy1, dx2, dy2 = det["sbbox"]
                    ix1, iy1 = max(cx1, dx1), max(cy1, dy1)
                    ix2, iy2 = min(cx2, dx2), min(cy2, dy2)
                    if ix2 <= ix1 or iy2 <= iy1:
                        continue
                    sub_c = cmask[iy1 - cy1 : iy2 - cy1, ix1 - cx1 : ix2 - cx1]
                    sub_d = det["mask"][iy1 - dy1 : iy2 - dy1, ix1 - dx1 : ix2 - dx1]
                    inter = int(np.logical_and(sub_c, sub_d).sum())
                    if inter >= 60 and float(inter) / float(det["area"]) >= 0.18:
                        overlaps.append((det["s_det"], di, det))

                if len(overlaps) >= 2:
                    overlaps.sort(key=lambda t: -t[0])
                    chosen_dets: list[dict[str, Any]] = []
                    for _sc, _di, cand_d in overlaps:
                        dx1, dy1, dx2, dy2 = cand_d["sbbox"]
                        disjoint_all = True
                        for prev in chosen_dets:
                            px1, py1, px2, py2 = prev["sbbox"]
                            ix1, iy1 = max(dx1, px1), max(dy1, py1)
                            ix2, iy2 = min(dx2, px2), min(dy2, py2)
                            if ix2 > ix1 and iy2 > iy1:
                                inter_dd = float(
                                    np.logical_and(
                                        cand_d["mask"][iy1 - dy1 : iy2 - dy1, ix1 - dx1 : ix2 - dx1],
                                        prev["mask"][iy1 - py1 : iy2 - py1, ix1 - px1 : ix2 - px1],
                                    ).sum()
                                )
                                iou_dd = inter_dd / max(1.0, float(cand_d["area"] + prev["area"]) - inter_dd)
                                if iou_dd >= 0.12:
                                    disjoint_all = False
                                    break
                        if disjoint_all:
                            chosen_dets.append(cand_d)

                    if len(chosen_dets) >= 2:
                        # Compute weighted Voronoi costs inside cmask across disjoint seeds
                        cost_stack = []
                        valid_seeds = 0
                        p_fg_c = p_fg[cy1:cy2, cx1:cx2]
                        for det in chosen_dets[:4]:
                            seed = np.zeros_like(cmask, dtype=bool)
                            dx1, dy1, dx2, dy2 = det["sbbox"]
                            ix1, iy1 = max(cx1, dx1), max(cy1, dy1)
                            ix2, iy2 = min(cx2, dx2), min(cy2, dy2)
                            seed[iy1 - cy1 : iy2 - cy1, ix1 - cx1 : ix2 - cx1] = (
                                det["mask"][iy1 - dy1 : iy2 - dy1, ix1 - dx1 : ix2 - dx1]
                                & cmask[iy1 - cy1 : iy2 - cy1, ix1 - cx1 : ix2 - cx1]
                            )
                            if int(seed.sum()) >= 15:
                                valid_seeds += 1
                                dmap = ndimage.distance_transform_edt(~seed).astype(np.float32) * (1.20 - p_fg_c)
                                cost_stack.append(dmap)
                        if valid_seeds >= 2:
                            owner = np.argmin(np.stack(cost_stack, axis=0), axis=0)
                            parts = []
                            for k_part in range(len(cost_stack)):
                                pm_k = cmask & (owner == k_part)
                                n_pk, l_pk, st_pk, _ = cv2.connectedComponentsWithStats(pm_k.astype(np.uint8), connectivity=8)
                                for sub_id in range(1, n_pk):
                                    sx_i, sy_i, sw_i, sh_i, sa_i = [int(v) for v in st_pk[sub_id]]
                                    if sa_i >= 20:
                                        sub_m = (l_pk[sy_i : sy_i + sh_i, sx_i : sx_i + sw_i] == sub_id)
                                        parts.append(((cx1 + sx_i, cy1 + sy_i, cx1 + sx_i + sw_i, cy1 + sy_i + sh_i), sub_m, False, False))
                            if len(parts) >= 2:
                                raw_comps.extend(parts)
                                split_done = True

            if not split_done:
                raw_comps.append(((cx1, cy1, cx2, cy2), cmask, False, False))

        # 2. Waist-Only Split Healing
        n_raw = len(raw_comps)
        parent = list(range(n_raw))

        def _find(u: int) -> int:
            while parent[u] != u:
                parent[u] = parent[parent[u]]
                u = parent[u]
            return u

        def _union(u: int, v: int) -> None:
            ru, rv = _find(u), _find(v)
            if ru != rv:
                parent[rv] = ru

        contours = [_contour_yx(m, bb[0], bb[1]) for bb, m, _, _ in raw_comps]
        comp_areas = [int(m.sum()) for _, m, _, _ in raw_comps]

        # Precompute which high-conf dets overlap which raw_comp by >= 45%
        comp_strong_dets: list[set[int]] = [set() for _ in range(n_raw)]
        for i, (bb, m, _, _) in enumerate(raw_comps):
            bx1, by1, bx2, by2 = bb
            ai = max(1, comp_areas[i])
            for di, det in enumerate(local_s1):
                if det["s_det"] < 0.35:
                    continue
                dx1, dy1, dx2, dy2 = det["sbbox"]
                ix1, iy1 = max(bx1, dx1), max(by1, dy1)
                ix2, iy2 = min(bx2, dx2), min(by2, dy2)
                if ix2 <= ix1 or iy2 <= iy1:
                    continue
                inter = int(
                    np.logical_and(
                        m[iy1 - by1 : iy2 - by1, ix1 - bx1 : ix2 - bx1],
                        det["mask"][iy1 - dy1 : iy2 - dy1, ix1 - dx1 : ix2 - dx1],
                    ).sum()
                )
                if float(inter) / float(ai) >= 0.45:
                    comp_strong_dets[i].add(di)

        bridges_by_root: dict[int, list[tuple[tuple[int, int, int, int], np.ndarray]]] = {}
        edges: list[tuple[int, int, tuple[int, int, int, int], np.ndarray]] = []
        for i in range(n_raw):
            ix1, iy1, ix2, iy2 = raw_comps[i][0]
            for j in range(i + 1, n_raw):
                jx1, jy1, jx2, jy2 = raw_comps[j][0]
                if max(ix1, jx1) - min(ix2, jx2) > 20 or max(iy1, jy1) - min(iy2, jy2) > 20:
                    continue
                # Veto if claimed by two distinct high-conf Stage-1 detections
                di_set, dj_set = comp_strong_dets[i], comp_strong_dets[j]
                if di_set and dj_set and not (di_set & dj_set):
                    # Check whether those separate dets are actually disjoint
                    veto = False
                    for a_di in di_set:
                        da = local_s1[a_di]
                        ax1, ay1, ax2, ay2 = da["sbbox"]
                        for b_dj in dj_set:
                            db = local_s1[b_dj]
                            bx1_, by1_, bx2_, by2_ = db["sbbox"]
                            ox1, oy1 = max(ax1, bx1_), max(ay1, by1_)
                            ox2, oy2 = min(ax2, bx2_), min(ay2, by2_)
                            if ox2 <= ox1 or oy2 <= oy1:
                                veto = True
                                break
                            inter_ab = float(
                                np.logical_and(
                                    da["mask"][oy1 - ay1 : oy2 - ay1, ox1 - ax1 : ox2 - ax1],
                                    db["mask"][oy1 - by1_ : oy2 - by1_, ox1 - bx1_ : ox2 - bx1_],
                                ).sum()
                            )
                            if inter_ab / max(1.0, float(da["area"] + db["area"]) - inter_ab) < 0.15:
                                veto = True
                                break
                        if veto:
                            break
                    if veto:
                        continue

                dist, pt_i, pt_j = _find_closest_points(contours[i], contours[j])
                if dist > 20.0:
                    continue

                lx1 = max(0, int(min(pt_i[1], pt_j[1])) - 2)
                ly1 = max(0, int(min(pt_i[0], pt_j[0])) - 2)
                lx2 = min(sw, int(max(pt_i[1], pt_j[1])) + 3)
                ly2 = min(sh, int(max(pt_i[0], pt_j[0])) + 3)
                line_m = np.zeros((ly2 - ly1, lx2 - lx1), dtype=np.uint8)
                cv2.line(
                    line_m,
                    (int(round(pt_i[1])) - lx1, int(round(pt_i[0])) - ly1),
                    (int(round(pt_j[1])) - lx1, int(round(pt_j[0])) - ly1),
                    color=1,
                    thickness=3,
                )
                lbool = line_m > 0
                if not lbool.any():
                    continue
                fg_line = float(p_fg[ly1:ly2, lx1:lx2][lbool].mean())
                sp_line = float(p_spine[ly1:ly2, lx1:lx2][lbool].mean())

                shared_strong = [d for d in (di_set & dj_set) if local_s1[d]["s_det"] >= 0.42]
                case_a = bool(shared_strong) and (fg_line >= 0.18)
                case_b = (dist <= 10.0) and (fg_line >= 0.28) and (sp_line >= 0.22)
                if case_a or case_b:
                    gate = 0.16 if case_a else 0.20
                    waist_m = lbool & (p_fg[ly1:ly2, lx1:lx2] >= gate)
                    edges.append((i, j, (lx1, ly1, lx2, ly2), waist_m))
                    _union(i, j)

        groups: dict[int, list[int]] = {}
        for i in range(n_raw):
            groups.setdefault(_find(i), []).append(i)
        for i, _j, lbb, wm in edges:
            bridges_by_root.setdefault(_find(i), []).append((lbb, wm))

        merged_comps: list[tuple[tuple[int, int, int, int], np.ndarray, bool, bool]] = []
        for root_id, idxs in groups.items():
            if len(idxs) == 1 and root_id not in bridges_by_root:
                merged_comps.append(raw_comps[idxs[0]])
                continue
            ux1 = min(raw_comps[k][0][0] for k in idxs)
            uy1 = min(raw_comps[k][0][1] for k in idxs)
            ux2 = max(raw_comps[k][0][2] for k in idxs)
            uy2 = max(raw_comps[k][0][3] for k in idxs)
            for lbb, _ in bridges_by_root.get(root_id, []):
                ux1, uy1 = min(ux1, lbb[0]), min(uy1, lbb[1])
                ux2, uy2 = max(ux2, lbb[2]), max(uy2, lbb[3])
            canvas = np.zeros((uy2 - uy1, ux2 - ux1), dtype=bool)
            for k in idxs:
                cx1, cy1, cx2, cy2 = raw_comps[k][0]
                canvas[cy1 - uy1 : cy2 - uy1, cx1 - ux1 : cx2 - ux1] |= raw_comps[k][1]
            for (lx1, ly1, lx2, ly2), wm in bridges_by_root.get(root_id, []):
                canvas[ly1 - uy1 : ly2 - uy1, lx1 - ux1 : lx2 - ux1] |= wm
            merged_comps.append(((ux1, uy1, ux2, uy2), canvas, True, False))

        # 3. Uncovered High-Confidence R-CNN Rescue
        for det in local_s1:
            if det["s_det"] < 0.48:
                continue
            dx1, dy1, dx2, dy2 = det["sbbox"]
            dm = det["mask"]
            ov_px = int(np.logical_and(dm, m_bin[dy1:dy2, dx1:dx2]).sum())
            if float(ov_px) / float(max(1, det["area"])) >= 0.20:
                continue
            res_m = dm & (p_fg[dy1:dy2, dx1:dx2] >= 0.20)
            ra = int(res_m.sum())
            if ra >= 28 and float(p_fg[dy1:dy2, dx1:dx2][res_m].mean()) >= 0.26:
                rbb = _mask_bbox_u8(res_m.astype(np.uint8))
                if rbb is not None:
                    rx1, ry1, rx2, ry2 = rbb
                    merged_comps.append((
                        (dx1 + rx1, dy1 + ry1, dx1 + rx2, dy1 + ry2),
                        res_m[ry1:ry2, rx1:rx2].copy(),
                        False,
                        True,
                    ))

        # 4. Compute orthogonal 17-d features + v_zero for every candidate
        cand_list: list[HybridCandidate] = []
        for (sx1, sy1, sx2, sy2), m_crop, is_healed, is_rescued in merged_comps:
            mc = m_crop > 0
            area = int(mc.sum())
            if area < 20:
                continue
            fg_vals = p_fg[sy1:sy2, sx1:sx2][mc]
            p_fg_mean = float(fg_vals.mean())
            p_fg_q85 = float(np.quantile(fg_vals, 0.85))
            p_fg_max = float(fg_vals.max())
            p_core_mean = float(p_core[sy1:sy2, sx1:sx2][mc].mean())
            p_spine_cov = float((p_spine[sy1:sy2, sx1:sx2][mc] >= 0.35).mean())
            peer_agree = 1.0 - float(p_disagree[sy1:sy2, sx1:sx2][mc].mean())

            # Boundary sharpness: inner perimeter p_fg minus 1-px outer ring p_fg
            px1, py1 = max(0, sx1 - 1), max(0, sy1 - 1)
            px2, py2 = min(sw, sx2 + 1), min(sh, sy2 + 1)
            pad_m = np.zeros((py2 - py1, px2 - px1), dtype=np.uint8)
            pad_m[sy1 - py1 : sy2 - py1, sx1 - px1 : sx2 - px1] = mc.astype(np.uint8)
            er_m = cv2.erode(pad_m, k3, iterations=1)
            dil_m = cv2.dilate(pad_m, k3, iterations=1)
            in_b = (pad_m > 0) & (er_m == 0)
            out_r = (dil_m > 0) & (pad_m == 0)
            fg_pad = p_fg[py1:py2, px1:px2]
            in_p = float(fg_pad[in_b].mean()) if in_b.any() else p_fg_mean
            out_p = float(fg_pad[out_r].mean()) if out_r.any() else 0.0
            boundary_drop = float(np.clip(in_p - out_p, -0.5, 1.0))

            s_det_best, det_iou_best, s1_view_frac = 0.0, 0.0, 0.0
            for det in local_s1:
                dx1, dy1, dx2, dy2 = det["sbbox"]
                ix1, iy1 = max(sx1, dx1), max(sy1, dy1)
                ix2, iy2 = min(sx2, dx2), min(sy2, dy2)
                if ix2 <= ix1 or iy2 <= iy1:
                    continue
                inter = float(
                    np.logical_and(
                        mc[iy1 - sy1 : iy2 - sy1, ix1 - sx1 : ix2 - sx1],
                        det["mask"][iy1 - dy1 : iy2 - dy1, ix1 - dx1 : ix2 - dx1],
                    ).sum()
                )
                if inter <= 0.0:
                    continue
                iou_d = inter / max(1.0, float(area + det["area"]) - inter)
                score_match = det["s_det"] * max(0.35, iou_d)
                if score_match > s_det_best * max(0.35, det_iou_best):
                    s_det_best = float(det["s_det"])
                    det_iou_best = float(iou_d)
                    s1_view_frac = float(det["view_agree"])

            d_multi_med = float(np.median(d_multi_sl[sy1:sy2, sx1:sx2][mc]))
            d_barb_med = float(np.median(d_barb_sl[sy1:sy2, sx1:sx2][mc]))
            t_rib_med = float(np.median(t_rib_sl[sy1:sy2, sx1:sx2][mc]))
            log_area = float(math.log1p(area))
            bw, bh = max(1, sx2 - sx1), max(1, sy2 - sy1)
            thinness = float(min(1.0, float(area) / float(bw * bh)))
            gx_c = float(x0) + 0.5 * (sx1 + sx2)
            gy_c = float(y0) + 0.5 * (sy1 + sy2)
            rho = math.hypot(gx_c - cx_s, gy_c - cy_s) / r_sun
            mu_cent = float(math.sqrt(max(0.0, 1.0 - rho * rho)))
            p_fg_x_agree = p_fg_mean * peer_agree

            t_rib_q90 = float(np.quantile(t_rib_sl[sy1:sy2, sx1:sx2][mc], 0.90))
            d_multi_q90 = float(np.quantile(d_multi_sl[sy1:sy2, sx1:sx2][mc], 0.90))
            snr_local = float(
                np.clip(
                    (float(bg_15[sy1:sy2, sx1:sx2][mc].mean()) - float(i_flat_f[sy1:sy2, sx1:sx2][mc].mean()))
                    / max(1.0, float(loc_sd_sl[sy1:sy2, sx1:sx2][mc].mean())),
                    -2.0,
                    12.0,
                )
            )
            if_pad = i_flat_f[py1:py2, px1:px2]
            in_i = float(if_pad[in_b].mean()) if in_b.any() else 0.0
            out_i = float(if_pad[out_r].mean()) if out_r.any() else in_i
            edge_contrast = float(np.clip((out_i - in_i) / 32.0, -1.0, 4.0))
            ys_m, xs_m = np.nonzero(mc)
            if len(xs_m) >= 3:
                dx_m = xs_m.astype(np.float64) - float(xs_m.mean())
                dy_m = ys_m.astype(np.float64) - float(ys_m.mean())
                cxx = float((dx_m * dx_m).mean())
                cyy = float((dy_m * dy_m).mean())
                cxy = float((dx_m * dy_m).mean())
                tr = cxx + cyy
                disc = math.sqrt(max(0.0, (cxx - cyy) ** 2 + 4.0 * cxy * cxy))
                l1 = max(1.0, 0.5 * (tr + disc))
                l2 = max(1.0, 0.5 * (tr - disc))
                elongation = float(min(25.0, math.sqrt(l1 / l2)))
            else:
                elongation = 1.0
            perim = max(1.0, float(in_b.sum()))
            circularity = float(min(1.0, 4.0 * math.pi * float(area) / (perim * perim)))
            pl_pad = plage_sl[py1:py2, px1:px2]
            plage_ring_q95 = float(np.quantile(pl_pad[out_r], 0.95)) if out_r.any() else 0.0

            feats = np.array(
                [
                    p_fg_mean, p_fg_q85, p_fg_max, p_core_mean, p_spine_cov,
                    peer_agree, boundary_drop, s_det_best, det_iou_best, s1_view_frac,
                    d_multi_med, d_barb_med, t_rib_med, log_area, thinness,
                    mu_cent, p_fg_x_agree,
                    t_rib_q90, d_multi_q90, snr_local, edge_contrast, elongation, circularity, plage_ring_q95,
                ],
                dtype=np.float32,
            )
            v_zero = (
                0.45 * p_fg_x_agree
                + 0.35 * (s_det_best * max(0.40, det_iou_best))
                + 0.20 * max(0.0, boundary_drop)
            )
            cand_list.append(
                HybridCandidate(
                    bbox=(x0 + sx1, y0 + sy1, x0 + sx2, y0 + sy2),
                    mask_u8=mc.astype(np.uint8),
                    features=feats,
                    v_zero=float(v_zero),
                    is_healed=is_healed,
                    is_rescued=is_rescued,
                    v_hat=float(v_zero),
                )
            )

        out_by_tau[tkey] = cand_list
    return out_by_tau


def _predict_metascorer(models: list[dict[str, np.ndarray]], feats: np.ndarray) -> float:
    acc = 0.0
    for m in models:
        xs = np.clip((feats.astype(np.float64) - m["mu"]) / m["sd"], -5.0, 5.0)
        xb = np.concatenate([[1.0], xs])
        y_r = float(np.clip(xb @ m["w_ridge"], 0.0, 1.0))
        z_l = float(np.clip(xb @ m["w_log"], -20.0, 20.0))
        p_l = 1.0 / (1.0 + math.exp(-z_l))
        acc += 0.65 * y_r + 0.35 * (0.72 * p_l)
    return acc / float(max(1, len(models)))


def _compose_disjoint_crops(
    cands: list[HybridCandidate],
    scores: list[float],
    keep_thresh: float,
    min_area: int,
) -> list[tuple[tuple[int, int, int, int], np.ndarray, int]]:
    """Rank candidates by descending utility score and resolve strict pairwise-disjoint crops."""
    selected = [
        (c, sc)
        for c, sc in zip(cands, scores)
        if sc >= keep_thresh and int(c.mask_u8.sum()) >= min_area
    ]
    if not selected:
        return []
    selected.sort(key=lambda item: -item[1])

    ux0 = min(c.bbox[0] for c, _ in selected)
    uy0 = min(c.bbox[1] for c, _ in selected)
    ux1 = max(c.bbox[2] for c, _ in selected)
    uy1 = max(c.bbox[3] for c, _ in selected)
    out: list[tuple[tuple[int, int, int, int], np.ndarray, int]] = []
    try:
        for idx, (c, sc) in enumerate(selected):
            bx1, by1, bx2, by2 = c.bbox
            ov = _OWNER_BUF[by1:by2, bx1:bx2]
            sv = _SCORE_BUF[by1:by2, bx1:bx2]
            mb = c.mask_u8 > 0
            # Higher-ranked candidate already claimed > 45% of this candidate -> suppress duplicate
            free_m = mb & (ov < 0)
            if int(free_m.sum()) < min_area or float(free_m.sum()) < 0.55 * float(mb.sum()):
                continue
            ov[free_m] = idx
            sv[free_m] = float(sc)

        for idx, (c, _sc) in enumerate(selected):
            bx1, by1, bx2, by2 = c.bbox
            inst_m = (_OWNER_BUF[by1:by2, bx1:bx2] == idx)
            area_c = int(inst_m.sum())
            if area_c < min_area:
                continue
            out.append((c.bbox, inst_m.copy(), area_c))
    finally:
        _OWNER_BUF[uy0:uy1, ux0:ux1] = -1
        _SCORE_BUF[uy0:uy1, ux0:ux1] = 0.0
    return out


def encode_crops_to_rles(crops: list[tuple[tuple[int, int, int, int], np.ndarray, int]]) -> list[dict]:
    rles: list[dict] = []
    for (bx1, by1, bx2, by2), inst_m, _area in crops:
        try:
            _TMP_MASK[by1:by2, bx1:bx2] = inst_m.astype(np.uint8)
            counts = mu.encode(_TMP_MASK)["counts"].decode()
            rles.append(referee_metric.rle(counts))
        finally:
            _TMP_MASK[by1:by2, bx1:bx2] = 0
    return rles


def load_metascorer_json(path: Path) -> tuple[dict[str, Any], list[dict[str, np.ndarray]]]:
    data = json.loads(path.read_text(encoding="utf-8"))
    models = [
        {
            "mu": np.asarray(m["mu"], dtype=np.float64),
            "sd": np.asarray(m["sd"], dtype=np.float64),
            "w_ridge": np.asarray(m["w_ridge"], dtype=np.float64),
            "w_log": np.asarray(m["w_log"], dtype=np.float64),
        }
        for m in data.get("models", [])
    ]
    return data["best_params"], models


def decode_test_stem_rles(
    cands: list[HybridCandidate],
    models: list[dict[str, np.ndarray]],
    keep_thresh: float,
    min_area: int,
) -> list[dict]:
    if not cands:
        return []
    scores = [_predict_metascorer(models, c.features) for c in cands] if models else [c.v_zero for c in cands]
    crops = _compose_disjoint_crops(cands, scores, keep_thresh=keep_thresh, min_area=min_area)
    return encode_crops_to_rles(crops)


def count_overlapping_stems(rles_by_stem: dict[str, list[dict]]) -> int:
    bad = 0
    for rles in rles_by_stem.values():
        if len(rles) < 2:
            continue
        inter = np.asarray(mu.iou(rles, rles, [1] * len(rles)), dtype=np.float64)
        np.fill_diagonal(inter, 0.0)
        if np.any(inter > 0.0):
            bad += 1
    return bad


def write_predictions_csv(rles_by_stem: dict[str, list[dict]], stems: list[str], out_csv: str | Path) -> Path:
    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8", newline="") as handle:
        w = csv.writer(handle)
        w.writerow(["filament_id", "segmentation_rle"])
        for stem in sorted(stems):
            idx_out = 0
            for r in rles_by_stem.get(stem, []):
                if not r or int(mu.area(r)) <= 0:
                    continue
                counts = r["counts"].decode() if isinstance(r["counts"], (bytes, bytearray)) else str(r["counts"])
                if not counts:
                    continue
                w.writerow([f"{stem}_{idx_out}", counts])
                idx_out += 1
    return out_path

In [ ]:
"""Cell 6: Offline Multi-Fold Streaming Inference & Kaggle Submission Generation."""


def _find_weights_dir() -> Path:
    env_w = os.environ.get("WEIGHTS_DIR", "").strip()
    if env_w and Path(env_w).is_dir():
        return Path(env_w)
    candidates = [
        Path("/kaggle/input/solar-filament-codeepnet-2026-weights"),
        Path("/kaggle/input/datasets/samiulislam180041221/solar-filament-codeepnet-2026-weights"),
        Path("./weights"),
        Path("../kaggle_weights_dataset"),
    ]
    for p in candidates:
        if (p / "metascorer_v10.json").is_file():
            return p
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.is_dir():
        for hit in sorted(kaggle_in.rglob("metascorer_v10.json")):
            return hit.parent
    raise FileNotFoundError("Could not locate metascorer_v10.json under WEIGHTS_DIR or /kaggle/input")


def _find_test_img_dir() -> Path:
    env_t = os.environ.get("TEST_IMG_DIR", "").strip()
    if env_t and Path(env_t).is_dir():
        return Path(env_t)
    candidates = [
        Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),
        Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),
        Path("/kaggle/input/filament-segmentation-2026/test/test_images"),
    ]
    for p in candidates:
        if p.is_dir():
            return p
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.is_dir():
        for p in sorted(kaggle_in.rglob("test_images")):
            if p.is_dir() and any(p.iterdir()):
                return p
    raise FileNotFoundError("Could not locate test_images directory under TEST_IMG_DIR or /kaggle/input")


def _resolve_output_csv() -> Path:
    env_o = os.environ.get("OUTPUT_CSV", "").strip()
    if env_o:
        return Path(env_o)
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working/submission.csv")
    return Path("submission.csv")


def _discover_fold_checkpoints(weights_dir: Path) -> list[tuple[int, Path | None, Path]]:
    nat_files: dict[int, Path] = {}
    for p in sorted(weights_dir.glob("codeepnet_native_f*_fp16.pt")):
        m = re.search(r"codeepnet_native_f(\d+)_fp16\.pt$", p.name)
        if m:
            nat_files[int(m.group(1))] = p
    if not nat_files:
        raise FileNotFoundError(f"No codeepnet_native_f*_fp16.pt weights found in {weights_dir}")
    pairs: list[tuple[int, Path | None, Path]] = []
    for k in sorted(nat_files):
        s1_p = weights_dir / f"stage1_mrcnn_f{k}_fp16.pt"
        pairs.append((k, s1_p if s1_p.is_file() else None, nat_files[k]))
    return pairs


def _validate_csv_contract(csv_path: Path, valid_stems: set[str]) -> tuple[int, int]:
    rows = 0
    stems_seen: set[str] = set()
    ids_seen: set[str] = set()
    with csv_path.open("r", encoding="utf-8", newline="") as fh:
        rdr = csv.reader(fh)
        header = next(rdr)
        assert header == ["filament_id", "segmentation_rle"], f"Unexpected header: {header}"
        for row in rdr:
            assert len(row) == 2, f"Malformed row length: {row}"
            fid, rle_str = row[0], row[1]
            assert fid not in ids_seen, f"Duplicate filament_id: {fid}"
            assert "_" in fid, f"Missing stem_idx separator: {fid}"
            stem = fid.rsplit("_", 1)[0]
            assert stem in valid_stems, f"Invalid test stem in submission: {stem}"
            assert isinstance(rle_str, str) and len(rle_str) > 0, f"Empty RLE for {fid}"
            assert int(mu.area(referee_metric.rle(rle_str))) > 0, f"Zero-area RLE for {fid}"
            ids_seen.add(fid)
            stems_seen.add(stem)
            rows += 1
    return rows, len(stems_seen)


def run_inference() -> Path:
    import time
    t0 = time.time()
    weights_dir = _find_weights_dir()
    test_dir = _find_test_img_dir()
    out_csv = _resolve_output_csv()
    max_imgs = int(os.environ.get("MAX_TEST_IMAGES", "0"))
    raw_tta = os.environ.get("TTA_VIEWS", "0,1,2,3").strip()
    tta_views = tuple(int(x) for x in raw_tta.split(",") if x.strip()) or (0, 1, 2, 3)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if device.type == "cuda" else 0.0
    keep_all_on_gpu = device.type == "cuda" and vram_gb >= 12.0

    best_params, fold_models = load_metascorer_json(weights_dir / "metascorer_v10.json")
    ckpt_pairs = _discover_fold_checkpoints(weights_dir)
    n_folds = len(ckpt_pairs)
    inv_folds = 1.0 / float(n_folds)

    sel_ts = float(best_params["tau_seed"])
    sel_dh = float(best_params["delta_hyst"])
    keep_thresh = float(best_params["keep_thresh"])
    min_area = int(best_params["min_area"])

    img_paths = sorted(
        [p for p in test_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")],
        key=lambda p: p.stem,
    )
    if max_imgs > 0:
        img_paths = img_paths[:max_imgs]
    test_stems = [p.stem for p in img_paths]

    print(
        f"[v10-offline] weights={weights_dir} (folds={[k for k, _, _ in ckpt_pairs]}) | "
        f"test_imgs={len(img_paths)} | device={device} ({vram_gb:.1f} GB) | "
        f"keep_on_gpu={keep_all_on_gpu} | tta={tta_views} | "
        f"params=(tau={sel_ts:.2f}, dh={sel_dh:.2f}, keep={keep_thresh:.2f}, min_area={min_area})",
        flush=True,
    )

    load_dev = device if keep_all_on_gpu else torch.device("cpu")
    loaded_folds: list[tuple[int, nn.Module | None, nn.Module]] = []
    for k, s1_p, nat_p in ckpt_pairs:
        s1_m = load_stage1_detector_from_fp16(s1_p, load_dev, det_size=DET_SIZE) if s1_p is not None else None
        nat_m = load_native_codeepnet_from_fp16(nat_p, load_dev)
        loaded_folds.append((k, s1_m, nat_m))

    n_s1_models = sum(1 for _k, s1_m, _ in loaded_folds if s1_m is not None)
    total_s1_views = max(1, n_s1_models * len(tta_views))
    test_rles: dict[str, list[dict[str, Any]]] = {}

    for idx, img_path in enumerate(img_paths, 1):
        stem = img_path.stem
        raw_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if raw_gray is None:
            raise RuntimeError(f"Failed to read test image: {img_path}")
        hw3_u8, geom = build_physics_channels_u8(raw_gray)
        x0, y0 = active_disk_slice_origin(float(geom["cx"]), float(geom["cy"]), slice_sz=SLICE_SIZE, img_size=IMG_SIZE)

        raw_s1_cands: list[dict[str, Any]] = []
        acc_f16: np.ndarray | None = None

        for fi, (_k, s1_m, nat_m) in enumerate(loaded_folds):
            if s1_m is not None:
                if not keep_all_on_gpu and device.type == "cuda":
                    s1_m.to(device)
                raw_s1_cands.extend(
                    predict_stage1_single(
                        s1_m,
                        hw3_u8,
                        geom,
                        device=device,
                        det_size=DET_SIZE,
                        tta_views=tta_views,
                        score_thresh=0.10,
                        view_offset=fi * len(tta_views),
                    )
                )
                if not keep_all_on_gpu and device.type == "cuda":
                    s1_m.to("cpu")

            if not keep_all_on_gpu and device.type == "cuda":
                nat_m.to(device)
            _nx0, _ny0, p_fg, p_core, p_spine, p_dis = predict_native_full_disk(
                nat_m,
                hw3_u8,
                geom,
                device=device,
                slice_size=SLICE_SIZE,
                tta_views=tta_views,
            )
            if not keep_all_on_gpu and device.type == "cuda":
                nat_m.to("cpu")

            cur = np.stack([p_fg, p_core, p_spine, p_dis], axis=0).astype(np.float32) * inv_folds
            if acc_f16 is None:
                acc_f16 = cur.astype(np.float16)
            else:
                acc_f16 = (acc_f16.astype(np.float32) + cur).astype(np.float16)

        assert acc_f16 is not None
        stack4 = acc_f16.astype(np.float32)
        s1_props = cluster_multiview_proposals(raw_s1_cands, iou_thr=0.45, n_views=total_s1_views)
        c_map = extract_stem_candidates_multitau(
            stem=stem,
            x0=x0,
            y0=y0,
            p_fg=stack4[0],
            p_core=stack4[1],
            p_spine=stack4[2],
            p_disagree=stack4[3],
            hw3_2048_u8=hw3_u8,
            geom=geom,
            s1_props=s1_props,
            tau_hyst_grid=((sel_ts, sel_dh),),
        )
        tkey = next(iter(c_map.keys()))
        test_rles[stem] = decode_test_stem_rles(
            c_map[tkey],
            models=fold_models,
            keep_thresh=keep_thresh,
            min_area=min_area,
        )

        if idx == 1 or idx % 10 == 0 or idx == len(img_paths):
            elapsed = (time.time() - t0) / 60.0
            eta = elapsed / float(idx) * float(len(img_paths) - idx)
            total_inst = sum(len(v) for v in test_rles.values())
            print(
                f"[v10-offline] {idx}/{len(img_paths)} ({stem}) | instances={total_inst} | "
                f"elapsed={elapsed:.2f}m | eta={eta:.2f}m",
                flush=True,
            )

    overlap_test = count_overlapping_stems(test_rles)
    assert overlap_test == 0, f"Detected overlapping masks in {overlap_test} stems!"

    write_predictions_csv(test_rles, test_stems, out_csv)
    n_rows, n_covered = _validate_csv_contract(out_csv, set(test_stems))
    print(
        f"[v10-offline] COMPLETE in {(time.time() - t0) / 60.0:.2f}m | csv={out_csv} | "
        f"rows={n_rows} | covered_stems={n_covered}/{len(test_stems)} | overlap_test={overlap_test}",
        flush=True,
    )
    return out_csv


if __name__ == "__main__":
    run_inference()